In [1]:
import sys
sys.path.insert(0, "..")

import mne
from pathlib import Path
from src.preprocessing.loader import load_raw
from src.preprocessing.filter import apply_filters
from src.preprocessing.epoching import make_epochs
from src.preprocessing.artifacts import mark_bad_channels, reject_by_amplitude, log_rejection
from src.preprocessing.ica import fit_ica, auto_detect_eog, apply_ica
from pipeline import load_config

cfg = load_config("../config.yaml")
print("Imports OK")

Imports OK


In [2]:
USE_SAMPLE_DATA = True  # flip to False (and set the path below) to run against a local recording

if USE_SAMPLE_DATA:
    sample_path = mne.datasets.sample.data_path()
    raw_path = sample_path / "MEG" / "sample" / "sample_audvis_raw.fif"
else:
    raw_path = Path(cfg["paths"]["raw_data"]) / "your_recording.fif"

raw = load_raw(str(raw_path))
# eog=True this time (M4 excluded it) -- fit_ica excludes EOG from the
# decomposition via picks="eeg", but auto_detect_eog needs the EOG channel
# present in `epochs` to correlate against. stim=True keeps the trigger
# channel needed for event-finding below. exclude=[] keeps EEG 053
# (already flagged bad in the recording's own metadata) around so this
# notebook's own mark_bad_channels call below can demonstrate on it, same
# as M4.
raw_eeg = raw.copy().pick_types(eeg=True, meg=False, stim=True, eog=True, exclude=[])
raw_eeg = mark_bad_channels(raw_eeg, bad_channels=["EEG 053"])

bp = cfg["preprocessing"]["bandpass"]
raw_eeg = apply_filters(raw_eeg, l_freq=bp["l_freq"], h_freq=bp["h_freq"], notch_freq=cfg["preprocessing"]["notch_freq"])

pp = cfg["preprocessing"]
epochs = make_epochs(raw_eeg, tmin=pp["epoch_tmin"], tmax=pp["epoch_tmax"])
epochs_clean = reject_by_amplitude(epochs, peak_to_peak_thresh={"eeg": 150e-6})
# reject_by_amplitude's threshold dict only checks the "eeg" key, so EOG
# passes through untouched.
log_rejection(epochs, epochs_clean)

print(f"Channel types present: {set(epochs_clean.get_channel_types())}")
print(f"Epochs ready for ICA: {len(epochs_clean)}")

Opening raw data file C:\Users\ke725\mne_data\MNE-sample-data\MEG\sample\sample_audvis_raw.fif...


    Read a total of 3 projection items:


        PCA-v1 (1 x 102)  idle


        PCA-v2 (1 x 102)  idle


        PCA-v3 (1 x 102)  idle


...

    Rejecting  epoch based on EEG : ['EEG 001', 'EEG 002', 'EEG 003', 'EEG 007']


    Rejecting  epoch based on EEG : ['EEG 001', 'EEG 002', 'EEG 003', 'EEG 007']


    Rejecting  epoch based on EEG : ['EEG 001', 'EEG 002', 'EEG 003', 'EEG 007']


31 bad epochs dropped


Epochs before rejection : 320
Epochs after rejection  : 289
Dropped                 : 31 (9.7%)
Kept                    : 289 (90.3%)
Channel types present: {'eeg', 'eog', 'stim'}
Epochs ready for ICA: 289


In [3]:
ica = fit_ica(epochs_clean, n_components=20, random_state=42)

print(f"ICA fitted with {ica.n_components_} components")

Fitting ICA to data using 59 channels (please be patient, this may take a while)


C:\Users\ke725\Documents\eeg-project\notebooks\..\src\preprocessing\ica.py:18: RuntimeWarning: The epochs you passed to ICA.fit() were baseline-corrected. However, we suggest to fit ICA only on data that has been high-pass filtered, but NOT baseline-corrected.
  ica.fit(epochs, picks="eeg")


Selecting by number: 20 components


Fitting ICA took 28.9s.


ICA fitted with 20 components


In [4]:
# Reference patterns: frontal-bilateral = eye, edge-concentrated = muscle,
# diffuse diagonal = cardiac, smooth/central = brain.
ica.plot_components(inst=epochs_clean);

C:\Users\ke725\Documents\eeg-project\.venv\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


**My Manual Read of the Components**

Looking at the topographies in Cell 4, **ICA002** is my primary guess for the eye-blink component. It appears as an extremely strong, tightly concentrated, symmetric blob right at the front-center of the scalp (roughly over Fpz), matching the frontal-bilateral pattern typical of blinks. Visually, it is the strongest and most sharply localized component in the whole set.

**ICA000** is a secondary candidate worth noting. It is also frontally weighted (reddish across the whole front half), but its gradient is broad and diffuse rather than sharply concentrated. This suggests a drift or reference artifact, or a diluted mix of frontal sources, rather than a clean single-source blink.

No other component in the grid stands out as edge-concentrated (muscle) or diffuse-diagonal (cardiac). Most of the remaining components, ICA010, ICA013, ICA014-018, appear lateralized toward one hemisphere or the temporal edges, which does not match the eye, muscle, or cardiac topography patterns. I am keeping this guess independent of any automated result; `auto_detect_eog` has not been run yet in this notebook, and the comparison will be addressed in a later session.

In [5]:
# plot_sources shows each component's activity over time, scrollable, the
# same way raw.plot() scrolls through channels -- except now each "channel"
# is a component's time course, not a physical electrode.
ica.plot_sources(epochs_clean)

Not setting metadata


289 matching events found


No baseline correction applied


0 projection items activated


Using matplotlib as 2D backend.


C:\Users\ke725\Documents\eeg-project\.venv\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


<MNEBrowseFigure size 800x800 with 4 Axes>

In [6]:
eog_indices = auto_detect_eog(ica, epochs)

print(f"Your manual guess: [ICA002, ICA000]")
print(f"Automated detection: {eog_indices}")

Using EOG channel: EOG 061


Auto-detected EOG component(s): [np.int64(0)]
Your manual guess: [ICA002, ICA000]
Automated detection: [np.int64(0)]


In [7]:
eog_indices_check, eog_scores = ica.find_bads_eog(epochs)
ica.plot_scores(eog_scores)

Using EOG channel: EOG 061


C:\Users\ke725\Documents\eeg-project\.venv\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


<Figure size 640x270 with 1 Axes>

In [8]:
ecg_indices, ecg_scores = ica.find_bads_ecg(epochs)
print(f"Auto-detected ECG-correlated component(s): {ecg_indices}")
ica.plot_scores(ecg_scores)

ValueError: Generating an artificial ECG channel can only be done for MEG data.

**Manual Guess:** ICA002 and ICA000

**Automated Detection:** ICA000

My guess was partially correct. I had chosen ICA000 as my runner‑up option, but after revising the component topographies I understood why ICA000 is considered the eye‑blink component: it shows a concentration of high‑power activity in the frontal region and is bilaterally distributed.

In [9]:
ica.plot_properties(epochs, picks=eog_indices)

    Using multitaper spectrum estimation with 7 DPSS windows


Not setting metadata


320 matching events found


No baseline correction applied


0 projection items activated


C:\Users\ke725\Documents\eeg-project\.venv\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


[<Figure size 700x600 with 6 Axes>]

In [10]:
# auto_detect_eog/find_bads_ecg above are both single-purpose, correlation-based
# detectors -- each only answers "does this correlate with one specific reference
# signal." ICLabel is a trained classifier that looks at each component's
# topography, spectrum, and time course together and assigns one of seven labels
# directly: brain, muscle artifact, eye blink, heart beat, line noise, channel
# noise, or other -- a genuinely different method, not just a third
# reference-channel check.
from mne_icalabel import label_components

ic_labels = label_components(epochs, ica, method="iclabel")
for idx, (label, proba) in enumerate(zip(ic_labels["labels"], ic_labels["y_pred_proba"])):
    marker = " <-- eog_indices" if idx in eog_indices else ""
    print(f"ICA{idx:03d}: {label:15s} (p={proba:.2f}){marker}")

C:\Users\ke725\AppData\Local\Temp\ipykernel_5492\1430728056.py:10: RuntimeWarning: The provided Epochs instance does not seem to be referenced to a common average reference (CAR). ICLabel was designed to classify features extracted from an EEG dataset referenced to a CAR (see the 'set_eeg_reference()' method for Raw and Epochs instances).
  ic_labels = label_components(epochs, ica, method="iclabel")
C:\Users\ke725\AppData\Local\Temp\ipykernel_5492\1430728056.py:10: RuntimeWarning: The provided Epochs instance is not filtered between 1 and 100 Hz. ICLabel was designed to classify features extracted from an EEG dataset bandpass filtered between 1 and 100 Hz (see the 'filter()' method for Raw and Epochs instances).
  ic_labels = label_components(epochs, ica, method="iclabel")
C:\Users\ke725\AppData\Local\Temp\ipykernel_5492\1430728056.py:10: RuntimeWarning: The provided ICA instance was fitted with a 'fastica' algorithm. ICLabel was designed with extended infomax ICA decompositions. To us

ICA000: eye blink       (p=0.86) <-- eog_indices
ICA001: brain           (p=1.00)
ICA002: brain           (p=0.69)
ICA003: brain           (p=0.50)
ICA004: brain           (p=0.50)
ICA005: brain           (p=0.94)
ICA006: brain           (p=0.99)
ICA007: brain           (p=0.90)
ICA008: brain           (p=0.99)
ICA009: other           (p=0.57)
ICA010: muscle artifact (p=0.70)
ICA011: other           (p=0.52)
ICA012: muscle artifact (p=0.95)
ICA013: muscle artifact (p=0.77)
ICA014: muscle artifact (p=0.95)
ICA015: muscle artifact (p=1.00)
ICA016: muscle artifact (p=0.96)
ICA017: muscle artifact (p=0.99)
ICA018: muscle artifact (p=0.90)
ICA019: brain           (p=0.62)


In [11]:
# Cell 8b's find_bads_ecg raised ValueError: this pipeline is EEG-only (no MEG
# channels), so MNE's artificial-ECG-proxy fallback has no signal to build from.
# There is no automated cardiac-artifact check available at all in this
# pipeline -- not a bug, a named limitation (see Cell 19 write-up).
ecg_indices = []  # no cardiac verification path exists for this EEG-only recording

print(f"ecg_indices set to {ecg_indices} (no MEG-based ECG proxy available)")

ecg_indices set to [] (no MEG-based ECG proxy available)


In [12]:
# Kept separate from eog_indices (and ecg_indices) so you can override the
# automated result here (e.g. final_exclude = [0]) without touching
# auto_detect_eog/find_bads_ecg themselves. Only fold ecg_indices in if
# Cell 8b's plot_scores and plot_properties actually supported it -- an
# empty ecg_indices ([]) is common and correct; don't force a merge just
# because the variable exists.
final_exclude = sorted(set(eog_indices) | set(ecg_indices))

print(f"Final component(s) being excluded: {final_exclude}")

Final component(s) being excluded: [np.int64(0)]


In [13]:
epochs_ica_clean = apply_ica(ica, epochs, exclude=final_exclude)

print(f"Original epochs shape : {epochs.get_data().shape}")
print(f"Cleaned epochs shape  : {epochs_ica_clean.get_data().shape}")

Applying ICA to Epochs instance


    Transforming to ICA space (20 components)


    Zeroing out 1 ICA component


    Projecting back using 59 PCA components


C:\Users\ke725\Documents\eeg-project\notebooks\..\src\preprocessing\ica.py:43: RuntimeWarning: The data you passed to ICA.apply() was baseline-corrected. Please note that ICA can introduce DC shifts, therefore you may wish to consider baseline-correcting the cleaned data again.
  ica.apply(epochs_clean)


Original epochs shape : (320, 70, 601)
Cleaned epochs shape  : (320, 70, 601)


In [14]:
import matplotlib.pyplot as plt
import numpy as np

epoch_idx = 0
ch_name = "EEG 001"  # a frontal channel -- most affected by a blink component
# If not in your channel list, print(epochs.ch_names) and pick another
# frontal one (lowest-numbered channels in this dataset's naming convention).

ch_idx = epochs.ch_names.index(ch_name)

before = epochs.get_data()[epoch_idx, ch_idx, :] * 1e6  # volts -> microvolts
after = epochs_ica_clean.get_data()[epoch_idx, ch_idx, :] * 1e6

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs.times, before, label="Before ICA", alpha=0.7)
ax.plot(epochs.times, after, label="After ICA", alpha=0.7)
ax.axvline(0, color="black", linestyle="--", linewidth=0.8, label="Stimulus onset")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (uV)")
ax.set_title(f"Before/After ICA -- {ch_name}, epoch {epoch_idx}")
ax.legend()
plt.show()

C:\Users\ke725\AppData\Local\Temp\ipykernel_5492\3457110683.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Averaging across all epochs is the same operation M6 will use to build
# ERPs -- running it here previews that logic AND shows the population-level
# effect of ICA cleaning, not just one lucky/unlucky trial.

evoked_before = epochs.average()
evoked_after = epochs_ica_clean.average()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(evoked_before.times, evoked_before.data[ch_idx] * 1e6, label="Before ICA")
ax.plot(evoked_after.times, evoked_after.data[ch_idx] * 1e6, label="After ICA")
ax.axvline(0, color="black", linestyle="--", linewidth=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude (uV)")
ax.set_title(f"Trial-averaged: Before/After ICA -- {ch_name}")
ax.legend()
plt.show()

C:\Users\ke725\AppData\Local\Temp\ipykernel_5492\4000821257.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# PSD comparison targets a different question than the time-domain plots
# above: did removing the component change the FREQUENCY content in the
# way we'd expect (low-frequency power, where blinks live, decreasing)?

fig_before = epochs.compute_psd(fmax=40).plot(picks=[ch_name], show=False)
fig_before.suptitle(f"PSD Before ICA -- {ch_name}")

fig_after = epochs_ica_clean.compute_psd(fmax=40).plot(picks=[ch_name], show=False)
fig_after.suptitle(f"PSD After ICA -- {ch_name}")

plt.show()

    Using multitaper spectrum estimation with 7 DPSS windows


Plotting power spectral density (dB=True).


Averaging across epochs before plotting...


Need more than one channel to make topography for eeg. Disabling interactivity.


    Using multitaper spectrum estimation with 7 DPSS windows


Plotting power spectral density (dB=True).


Averaging across epochs before plotting...


Need more than one channel to make topography for eeg. Disabling interactivity.


C:\Users\ke725\AppData\Local\Temp\ipykernel_5492\1619517132.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# Confirms ICA changed signal VALUES only, not epoch count/channels/labels --
# later modules need to trust that structure survived this cleaning step.
assert len(epochs) == len(epochs_ica_clean), "Epoch count changed unexpectedly!"
assert epochs.ch_names == epochs_ica_clean.ch_names, "Channel list changed unexpectedly!"
assert (epochs.events == epochs_ica_clean.events).all(), "Event labels changed unexpectedly!"

print("All structural checks passed -- ICA changed signal VALUES only, not shape/labels.")

All structural checks passed -- ICA changed signal VALUES only, not shape/labels.


In [18]:
from src.preprocessing.filter import save_processed

save_processed(epochs_ica_clean, "../data/processed/sub-01-ica-clean-epo.fif")
print("Saved ICA-cleaned epochs to data/processed/sub-01-ica-clean-epo.fif")

Overwriting existing file.


Overwriting existing file.


Overwriting existing file.


Saved ICA-cleaned epochs to data/processed/sub-01-ica-clean-epo.fif


## Excluded Component(s) — Topography Description

**Component(s) excluded:** ICA000 (`final_exclude = [0]`, Cell 12)

**Topography:** Symmetric, strongly concentrated at the front-center of the scalp (roughly over Fpz), fading smoothly toward posterior electrodes — the classic frontal-bilateral pattern `M5_concepts.md` §5 describes for eye blinks.

**Power spectrum:** Power concentrated at low frequencies, dropping off toward higher frequencies with only a modest bump near ~10 Hz — consistent with a blink (a slow, broad deflection), not a fast oscillation or muscle (EMG) contamination.

**Epochs image / averaged time course:** The per-trial banding in `plot_properties`' epochs-image panel is scattered rather than tightly locked to the stimulus-onset line, and the trial-averaged time course stays relatively flat with only a modest dip shortly after onset — consistent with blinks not being time-locked to the stimulus, so they partially (not fully) cancel out on average.

## Manual Detection vs. Automated Detection — Final Verdict

**My independent manual guess (before running the detector, Cell 5):** ICA002 (primary), ICA000 (secondary)
**Automated `auto_detect_eog` result (Cell 7):** ICA000
**Agreement:** Partial — my secondary pick matched the automated result; my primary pick did not.

My independent guess leaned on visual salience: ICA002 was the strongest, most tightly concentrated blob in the topography grid, so I picked it as the eye blink and treated ICA000's broader, more diffuse frontal pattern as a weaker secondary candidate. The automated correlation-based detector (`auto_detect_eog`, correlating each component against the real EOG channel) flagged only ICA000. Part 5's deeper inspection (`plot_properties`) supported ICA000: its power spectrum was low-frequency-dominant and its epochs-image banding was non-time-locked, both consistent with a blink. The ICLabel cross-check below independently confirmed ICA000 as "eye blink" (86% confidence) while classifying my top pick, ICA002, as "brain" (69% confidence) instead. This is a direct, concrete illustration of `M5_concepts.md` §6's caution that correlation-based (and by extension, purely visual) detection is "trustworthy but not infallible" — but here it cuts the other way: the strongest-looking, most visually salient component was not the artifact, and two independent methods (a real reference channel, and a classifier trained on independent data) converged on the *other* component instead. Visual salience alone was a misleading heuristic in this case.

## Cardiac (ECG) and ICLabel Cross-Check

**ECG check:** `find_bads_ecg()`'s no-real-channel fallback requires MEG
data to synthesize a proxy; this pipeline is EEG-only by design (M1), so
no automated cardiac-artifact check is possible here at all — not even an
approximation. `final_exclude` reflects this honestly (`ecg_indices = []`
by necessity, not by a clean result).

**ICLabel cross-check:** ICLabel labeled ICA000 "eye blink" at 86% confidence, agreeing with `auto_detect_eog`. It additionally labeled seven components (ICA010, ICA012–ICA018) as "muscle artifact" with confidences from 70% to 100%, and the remaining components mostly as "brain" (ICA001, ICA003–ICA008, ICA019) or "other" (ICA009, ICA011). None were labeled "heart beat." The muscle-artifact flags are notable: they were not caught by any other method in this notebook (no correlation-based muscle detector was run), so ICLabel surfaced a real category of potential contamination this pipeline would otherwise have missed entirely — though per `M5_concepts.md`'s caveat about ICLabel, these were not independently verified against a ground-truth muscle channel and were not excluded from `final_exclude`, since only the EOG-corroborated component had independent verification.

**What this means for the pipeline's artifact-rejection story:** blinks
are the only artifact type here with an independently verifiable removal
(a real recorded EOG channel to correlate against) — muscle and cardiac
component decisions rely entirely on ICA's own diagnostics (topography,
spectrum) or ICLabel's classifier output, with no ground-truth channel to
check against, and (for cardiac specifically) not even MNE's usual proxy
fallback. This is a real, named limitation, not a gap being glossed over.

## Before/After Comparison Summary

Cell 14's single-epoch overlay (epoch 0, channel EEG 001) showed a large difference: the pre-ICA signal peaked at ~130 µV, and removing ICA000 accounted for ~128 µV of that (i.e., epoch 0 contained a real blink, almost entirely attributable to the excluded component). Cell 15's trial-averaged overlay showed a much smaller effect — ~5 µV peak before ICA, ~2 µV of that removed by cleaning — confirming the prediction that averaging across many non-time-locked blinks partially cancels the artifact even before ICA is applied, though a smaller residual bias remains in the average. Cell 16's PSD comparison quantified the frequency-domain effect: power in the 1–5 Hz band (where blink energy concentrates) dropped by ~63% after ICA, while power in the 8–30 Hz alpha/beta band (genuine brain oscillations) changed by less than 1%. Together, this is direct evidence that ICA removed the blink-specific contamination without indiscriminately damping the rest of the EEG signal.